# PyRIT Scan

`pyrit_scan` is the primary command-line tool for running automated security assessments and red teaming attacks against AI systems. It leverages [scenarios](../code/scenarios/0_scenarios.ipynb) to define attack strategies and supports flexible [configuration](../code/setup/1_configuration.ipynb) for targeting different AI endpoints.

For configuration setup, see [Configuration](../getting_started/configuration.md).

For scenario-specific examples, see [AIRT](airt.ipynb), [Foundry](foundry.ipynb), and [Garak](garak.ipynb).

Note in this doc the ! prefaces all commands in the terminal so we can run in a Jupyter Notebook.

## Quick Start

For help:

In [ ]:
!pyrit_scan --help

usage: pyrit_scan [-h] [--server-url SERVER_URL] [--start-server]
                  [--stop-server] [--config-file CONFIG_FILE]
                  [--log-level LOG_LEVEL] [--request-timeout REQUEST_TIMEOUT]
                  [--list-scenarios] [--list-initializers] [--list-targets]
                  [--list-converters] [--list-datasets]
                  [--add-initializer FILE [FILE ...]] [--target TARGET]
                  [--initializers INITIALIZERS [INITIALIZERS ...]]
                  [--strategies SCENARIO_STRATEGIES [SCENARIO_STRATEGIES ...]]
                  [--max-concurrency MAX_CONCURRENCY]
                  [--max-retries MAX_RETRIES] [--memory-labels MEMORY_LABELS]
                  [--dataset-names DATASET_NAMES [DATASET_NAMES ...]]
                  [--max-dataset-size MAX_DATASET_SIZE]
                  [scenario_name]

PyRIT Scanner - Run AI security scenarios from the command line.

Requires a running PyRIT backend server. Use --start-server to launch one,
or connect

### Discovery

List all available scenarios:

In [ ]:
!pyrit_scan --list-scenarios


Error: Server not available at http://localhost:8000
Hint: Use '--start-server' to launch a local backend, or pass '--server-url <url>'.


**Tip**: You can also discover user-defined scenarios by providing initialization scripts:

```shell
pyrit_scan --list-scenarios --initialization-scripts ./my_custom_initializer.py
```

This will load your custom scenario definitions and include them in the list.

## Initializers

PyRITInitializers are how you can configure the CLI scanner. PyRIT includes several built-in initializers you can use with the `--initializers` flag.

The `--list-initializers` command shows all available initializers. Initializers are referenced by their filename (e.g., `target`, `scorer`, `simple`) regardless of which subdirectory they're in.

List the available initializers using the --list-initializers flag.

In [ ]:
!pyrit_scan --list-initializers


Error: Server not available at http://localhost:8000
Hint: Use '--start-server' to launch a local backend, or pass '--server-url <url>'.


### Running Scenarios

You need a single scenario to run, you need two things:

1. A Scenario. Many are defined in `pyrit.scenario.scenarios`. But you can also define your own in initialization_scripts.
2. Initializers (which can be supplied via `--initializers` or `--initialization-scripts` or `initializers` section of config file (see [here](../getting_started/pyrit_conf.md))). Scenarios often don't need many arguments, but they can be configured in different ways. And at the very least, most need an `objective_target` (the thing you're running a scan against) which you can configure by using the `--target` flag if your initializer registers targets (e.g. `target` initializer)
3. Scenario Strategies (optional). These are supplied by the `--scenario-strategies` flag and tell the scenario what to test, but they are always optional. Also note you can obtain these by running `--list-scenarios`

Basic usage will look something like:

```shell
pyrit_scan <scenario> --target <target_name> --initializers <initializer1> <initializer2> --scenario-strategies <strategy1> <strategy2>
```

You can also override scenario parameters directly from the CLI:

```shell
pyrit_scan <scenario> --max-concurrency 10 --max-retries 3 --memory-labels '{"experiment": "test1", "version": "v2"}'
```

Or concretely:

```shell
!pyrit_scan foundry.red_team_agent --target openai_chat --initializers target --scenario-strategies base64
```

Example with a basic configuration that runs the Foundry scenario against the objective target defined in the `target` initializer.

In [ ]:
!pyrit_scan foundry.red_team_agent --target openai_chat --initializers target --strategies base64


Error: Server not available at http://localhost:8000
Hint: Use '--start-server' to launch a local backend, or pass '--server-url <url>'.


Or with all options and multiple strategies:

```shell
pyrit_scan foundry.red_team_agent --target openai_chat --initializers target --strategies easy crescendo
```

You can also override scenario execution parameters:

```shell
# Override concurrency and retry settings
pyrit_scan foundry.red_team_agent --target openai_chat --initializers target --max-concurrency 10 --max-retries 3

# Add custom memory labels for tracking (must be valid JSON)
pyrit_scan foundry.red_team_agent --target openai_chat --initializers target --memory-labels '{"experiment": "test1", "version": "v2", "researcher": "alice"}'
```

Available CLI parameter overrides:
- `--max-concurrency <int>`: Maximum number of concurrent attack executions
- `--max-retries <int>`: Maximum number of automatic retries if the scenario raises an exception
- `--memory-labels <json>`: Additional labels to apply to all attack runs (must be a JSON string with string keys and values)

You can also use custom initialization scripts by passing file paths. It is relative to your current working directory, but to avoid confusion, full paths are always better:

```shell
pyrit_scan garak.encoding --initialization-scripts ./my_custom_config.py
```

#### Attaching Converters to a Technique

Strategies (techniques) can have a registered converter instance appended to them with the
`<technique>:converter.<name>` syntax. The converter is added to the request side of every attack
the technique produces, on top of any converters the technique already bakes in. This also works on
aggregate strategies (the converter is applied to every technique the aggregate expands to).

First discover the registered converter instances with `--list-converters` (converters are
registered by initializers, so pass the same `--initializers`/`--initialization-scripts` you use to run):

```shell
pyrit_scan --list-converters --initializers my_converters
```

Then reference a converter by name in `--strategies`:

```shell
# Add the registered "translation_spanish" converter to role_play only
pyrit_scan airt.rapid_response --target openai_chat --initializers load_default_datasets target my_converters --strategies role_play:converter.translation_spanish

# Chain multiple converters (applied in order) and combine with plain strategies
pyrit_scan airt.rapid_response --target openai_chat --initializers load_default_datasets target my_converters --strategies role_play:converter.translation_spanish:converter.base64 many_shot
```

#### Using Custom Scenarios

You can define your own scenarios in initialization scripts. The CLI will automatically discover any `Scenario` subclasses and make them available:


In [ ]:
# my_custom_scenarios.py

from pyrit.common import apply_defaults
from pyrit.prompt_target.openai.openai_chat_target import OpenAIChatTarget
from pyrit.scenario import DatasetAttackConfiguration, Scenario, ScenarioStrategy
from pyrit.score import SelfAskRefusalScorer, TrueFalseInverterScorer
from pyrit.setup import initialize_pyrit_async


class MyCustomStrategy(ScenarioStrategy):
    """Strategies for my custom scenario."""

    ALL = ("all", {"all"})
    Strategy1 = ("strategy1", set[str]())
    Strategy2 = ("strategy2", set[str]())


class MyCustomScenario(Scenario):
    """My custom scenario that does XYZ."""

    @apply_defaults
    def __init__(self, *, scenario_result_id=None, **kwargs):
        # Scenario-specific configuration only - no runtime parameters
        super().__init__(
            name="My Custom Scenario",
            version=1,
            objective_scorer=TrueFalseInverterScorer(scorer=SelfAskRefusalScorer(chat_target=OpenAIChatTarget())),
            strategy_class=MyCustomStrategy,
            default_strategy=MyCustomStrategy.ALL,
            default_dataset_config=DatasetAttackConfiguration(dataset_names=["harmbench"]),
            scenario_result_id=scenario_result_id,
        )
        # ... your scenario-specific initialization code

    async def _build_atomic_attacks_async(self, *, context):
        # The single abstract extension point every scenario implements.
        # Read runtime inputs from `context`; return the list of AtomicAttack to run.
        # Matrix-shaped scenarios can delegate to build_matrix_atomic_attacks(context=...).
        # Example: create attacks for each strategy composite
        return []


await initialize_pyrit_async(memory_db_type="InMemory")  # type: ignore
MyCustomScenario()

./AppData/Local/miniconda3/Lib/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


Found default environment files: ['./.pyrit/.env', './.pyrit/.env.local']
Loaded environment file: ./.pyrit/.env
Loaded environment file: ./.pyrit/.env.local


[pyrit:alembic] No new upgrade operations detected.


Then discover and run it:

```shell
# List to see it's available
pyrit_scan --list-scenarios --initialization-scripts ./my_custom_scenarios.py

# Run it with parameter overrides
pyrit_scan my_custom_scenario --initialization-scripts ./my_custom_scenarios.py --max-concurrency 10
```

The scenario name is automatically converted from the class name (e.g., `MyCustomScenario` becomes `my_custom_scenario`).